# Running parallel simulations

In [1]:
import numpy as np
from joblib import Parallel, delayed
from lucifex.mesh import rectangle_mesh, mesh_boundary
from lucifex.fdm import CN, FunctionSeries, ConstantSeries
from lucifex.solver import ibvp, BoundaryConditions
from lucifex.sim import Simulation, configure_simulation, run
from lucifex.fe2py import GridSimulation, as_grid_simulation
from lucifex.pde.diffusion import diffusion

@configure_simulation(
    store_delta=1,
)
def create_simulation(
    Nx: int,
    Ny: int,
    Lx: float,
    Ly: float,
    dt: float,
    d: float,
) -> Simulation:
    mesh = rectangle_mesh(Lx, Ly, Nx, Ny)
    boundary = mesh_boundary(
        mesh, 
        {
            "left": lambda x: x[0],
            "right": lambda x: x[0] - Lx,
            "lower": lambda x: x[1],
            "upper": lambda x: x[1] - Ly,
        },
    )
    t = ConstantSeries(mesh, name='t', ics=0.0)
    ics = lambda x: np.exp(-((x[0] - Lx/2)**2 + (x[1] - Ly/2)**2)/ (0.01 * Lx))
    bcs = BoundaryConditions(
        ("dirichlet", boundary.union, 0.0),  
    )
    u = FunctionSeries((mesh, 'P', 1), name='u', store=1)
    u_solver = ibvp(diffusion, ics, bcs)(u, dt, d, CN)
    return u_solver, t, dt


N_PROC = 3
STORE = 1
NX = 10
NY = 10
DT = 0.01
N_STOP = 10

def joblib_run(
    d: float,
    **run_kwargs
) -> GridSimulation:
    sim = create_simulation(store_delta=STORE)(NX, NY, 1.0, 1.0, DT, d)
    if run_kwargs:
        run(sim, **run_kwargs)
    return as_grid_simulation(sim)


d_opts = (10, 20, 30)

simulations = Parallel(n_jobs=N_PROC, return_as='generator')(
    delayed(joblib_run)(d, n_stop=N_STOP) for d in d_opts
)





ImportError: cannot import name 'Simulation' from partially initialized module 'lucifex.sim' (most likely due to a circular import) (/Users/George/Desktop/LUCiFEx/lucifex/sim/__init__.py)